In [ ]:
## run the pipelien 

# Optional: wipe all prior multi-size results and VCFs
rm -f data/vcf/del_*.{vcf,vcf.gz,vcf.gz.tbi}
rm -f results/del_*k31.freqk.*
rm -rf results/*bp_noerr


sbatch scripts/04_run_shorts_freqtest_all_noerr.sh
sbatch scripts/05_make_vcf_all.sh
sbatch scripts/06_freqk_index_all.sh
sbatch scripts/07_freqk_dedup_count_call_all_noerror.sh

In [19]:
import pandas as pd
import os

In [20]:
import os
import glob
from pathlib import Path

# Base results directory
results_dir = Path('../results')

# Find all directories ending with 'noerr'
noerr_dirs = sorted([d for d in results_dir.iterdir() if d.is_dir() and d.name.endswith('_noerr')])

print(f"Found {len(noerr_dirs)} noerr directories:\n")

# Collect all allele frequency files
allele_freq_files = []

for noerr_dir in noerr_dirs:
    # Look for .allele_frequencies.k31.tsv files
    freq_files = list(noerr_dir.glob('*.allele_frequencies.k31.tsv'))
    
    if freq_files:
        for f in freq_files:
            allele_freq_files.append(f)
            print(f"{noerr_dir.name:25s} -> {f.name}")
    else:
        print(f"{noerr_dir.name:25s} -> NO ALLELE FREQ FILE")

print(f"\n\nTotal allele frequency files found: {len(allele_freq_files)}")

# Show the full paths
print("\n" + "="*80)
print("Full paths:")
print("="*80)
for f in allele_freq_files:
    print(f)

FileNotFoundError: [Errno 2] No such file or directory: '../results'

In [18]:
for i in allele_freq_files:
    print(i)
    result = pd.read_csv(i)
    print(result)

../results/100bp_noerr/freq_100bp_050_noerr.allele_frequencies.k31.tsv
Empty DataFrame
Columns: [0.4633945|0.53660554]
Index: []
../results/10kb_noerr/freq_10kb_050_noerr.allele_frequencies.k31.tsv
Empty DataFrame
Columns: [0.36771673|0.6322833]
Index: []
../results/1kb_noerr/freq_1kb_050_noerr.allele_frequencies.k31.tsv
Empty DataFrame
Columns: [0.71232307|0.28767687]
Index: []
../results/500bp_noerr/freq_500bp_050_noerr.allele_frequencies.k31.tsv
Empty DataFrame
Columns: [0.518572|0.48142806]
Index: []
../results/5kb_noerr/freq_5kb_050_noerr.allele_frequencies.k31.tsv
Empty DataFrame
Columns: [0.5902227|0.40977725]
Index: []


In [33]:
os.listdir('/home/')

[]

In [30]:
import os
import glob
from pathlib import Path

# Base results directory
# New layout: results/{sv_type}/{cov_err}/{SV_TYPE}/{size}/{freq}/{k}/
results_dir = Path('../results')

# Find all allele frequency files recursively (any k value)
allele_freq_files = sorted(results_dir.rglob('*.allele_frequencies.k*.tsv'))

print(f"Found {len(allele_freq_files)} allele frequency files:\n")
print(f"{'SV':4s}  {'cov/err':12s}  {'size':6s}  {'freq':4s}  {'k':4s}  file")
print("-" * 75)

for f in allele_freq_files:
    parts = f.relative_to(results_dir).parts
    # parts: (sv_type, cov_err, SV_TYPE, size, freq, k, filename)
    sv_type = parts[2]   # 'DEL' or 'INS'
    cov_err = parts[1]   # 'cov50_err0'
    size    = parts[3]   # '100bp', '1kb', ...
    freq    = parts[4]   # 'f50'
    k       = parts[5]   # 'k31'
    print(f"{sv_type:4s}  {cov_err:12s}  {size:6s}  {freq:4s}  {k:4s}  {f.name}")

print(f"\nTotal: {len(allele_freq_files)} files")
print("\n" + "="*80)
print("Full paths:")
print("="*80)
for f in allele_freq_files:
    print(f)


Found 0 allele frequency files:

SV    cov/err       size    freq  k     file
---------------------------------------------------------------------------

Total: 0 files

Full paths:


In [ ]:
# ── Compute error metrics ──────────────────────────────────────────────────────
results_df["error"]     = results_df["af_alt"] - results_df["freq_nominal"]   # signed bias
results_df["abs_error"] = results_df["error"].abs()
results_df["rel_error"] = results_df["abs_error"] / results_df["freq_nominal"]

results_df[["sv_type","coverage","error_rate","size","freq_nominal","af_alt","error","abs_error"]].head(10)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

# ── Aesthetics ─────────────────────────────────────────────────────────────────
SIZE_ORDER  = ["100bp", "500bp", "1kb", "5kb", "10kb"]
SIZE_COLORS = dict(zip(SIZE_ORDER, ["#4e79a7", "#f28e2b", "#59a14f", "#e15759", "#76b7b2"]))
ERR_MARKERS = {}   # filled in dynamically below

sv_types   = ["DEL", "INS"]
coverages  = sorted(results_df["coverage"].unique())
error_rates = sorted(results_df["error_rate"].unique())

# Assign markers to error rates
_markers = ["o", "s", "^", "D"]
for er, mk in zip(error_rates, _markers):
    ERR_MARKERS[er] = mk

# ── Layout: rows = sv_type, cols = coverage ────────────────────────────────────
nrows, ncols = len(sv_types), len(coverages)
fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(3.6 * ncols, 3.6 * nrows),
    sharex=True, sharey=True,
    constrained_layout=True,
)
if nrows == 1: axes = [axes]
if ncols == 1: axes = [[ax] for ax in axes]

# Determine a shared axis range that includes the identity line
all_vals = pd.concat([results_df["freq_nominal"], results_df["af_alt"]])
pad = 0.06
lo, hi = all_vals.min() - pad, all_vals.max() + pad

for i, svt in enumerate(sv_types):
    for j, cov in enumerate(coverages):
        ax = axes[i][j]
        subset = results_df[(results_df["sv_type"] == svt) & (results_df["coverage"] == cov)]

        for err in error_rates:
            for size in SIZE_ORDER:
                pts = subset[(subset["error_rate"] == err) & (subset["size"] == size)]
                if pts.empty:
                    continue
                ax.scatter(
                    pts["freq_nominal"], pts["af_alt"],
                    color=SIZE_COLORS.get(size, "grey"),
                    marker=ERR_MARKERS.get(err, "x"),
                    s=75, alpha=0.88,
                    edgecolors="white", linewidths=0.4,
                    zorder=3,
                )

        # Identity line (perfect estimation)
        ax.plot([lo, hi], [lo, hi], color="black", lw=1, ls="--", alpha=0.35, zorder=1)

        # RMSE annotation
        if not subset.empty:
            rmse = np.sqrt(((subset["af_alt"] - subset["freq_nominal"]) ** 2).mean())
            ax.text(0.97, 0.03, f"RMSE = {rmse:.3f}",
                    transform=ax.transAxes, ha="right", va="bottom",
                    fontsize=7.5, color="0.35",
                    bbox=dict(fc="white", ec="none", alpha=0.7, pad=1))

        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)
        ax.set_aspect("equal")
        ax.set_title(f"{svt}  |  {cov}x coverage", fontsize=9, fontweight="bold")
        if j == 0:
            ax.set_ylabel("Estimated frequency  (af_alt)", fontsize=8)
        if i == nrows - 1:
            ax.set_xlabel("True frequency", fontsize=8)
        ax.tick_params(labelsize=7)

# ── Legends ────────────────────────────────────────────────────────────────────
size_patches = [mpatches.Patch(color=SIZE_COLORS[s], label=s) for s in SIZE_ORDER]
err_handles  = [
    mlines.Line2D([], [], color="0.4",
                  marker=ERR_MARKERS[e], linestyle="None",
                  markersize=7, label=f"error rate = {e}")
    for e in error_rates
]
fig.legend(
    handles=size_patches + [mpatches.Patch(visible=False)] + err_handles,
    title="SV size                                    Sequencing error",
    loc="lower center",
    ncol=len(SIZE_ORDER) + 1 + len(error_rates),
    fontsize=8,
    title_fontsize=8,
    bbox_to_anchor=(0.5, -0.06),
    frameon=False,
)

fig.suptitle("True vs estimated SV allele frequency  —  freqk benchmark",
             fontsize=11, fontweight="bold")

# ── Save ───────────────────────────────────────────────────────────────────────
Path("plots").mkdir(exist_ok=True)
fig.savefig("plots/true_vs_estimated.pdf", bbox_inches="tight")
fig.savefig("plots/true_vs_estimated.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved to plots/true_vs_estimated.{pdf,png}")
